# Q20.12 V3 supplementary board and CPU tests
English-language reproduction copy, not an executed test record. Original Untitled3.ipynb and downloaded ZIPs are preserved unchanged.
FPGA: 14 distinct supplied images, exact fixed-point reference comparison. CPU: scalar single-thread C++ compiled with GCC -O3, Butterfly only. Timing boundaries differ.
Enable switches only when deliberately repeating tests on the KV260. Do not run concurrent board notebooks.


In [ ]:
from pathlib import Path
from zipfile import ZipFile
import hashlib
import json
import shutil
import sys

# Run from the Jupyter Project directory on the KV260.
# These switches are intentionally disabled. This copy has NOT been executed.
RUN_BOARD = False
RUN_CPU = False
kit = Path("q20_12_return_board_check")
if not kit.is_dir():
    with ZipFile("q20_12_return_board_check.zip") as archive:
        for name in archive.namelist():
            resolved = Path(name).resolve()
            if not resolved.is_relative_to(Path.cwd().resolve()):
                raise ValueError("Unsafe archive path: " + name)
        if archive.testzip() is not None:
            raise ValueError("Archive CRC check failed")
        archive.extractall(".")
sys.path.insert(0, str(kit.resolve()))
import supplement_board

def check_kit_cross_platform(path):
    root = Path(path).resolve()
    hashes = json.loads((root / "kit_checksums.json").read_text())
    for name, expected in hashes.items():
        file = (root / name.replace("\\", "/")).resolve()
        if not file.is_relative_to(root):
            raise ValueError("Path escapes kit directory: " + name)
        if not file.is_file():
            raise FileNotFoundError("Missing kit file: " + name)
        if hashlib.sha256(file.read_bytes()).hexdigest() != expected:
            raise ValueError("Checksum mismatch: " + name)
    return root

# Normalize Windows manifest separators while retaining all integrity checks.
supplement_board.check_kit = check_kit_cross_platform
check_kit_cross_platform(kit)
if RUN_BOARD:
    output_dir = supplement_board.run_images(Path("srcnn_q20_12_v3_resetfix"), kit)
    board_zip = shutil.make_archive(str(output_dir), "zip", root_dir=str(output_dir))
    print("Board results archive:", board_zip)
if RUN_CPU:
    cpu_output_dir = supplement_board.run_cpu(kit, case="butterfly", repeats=3)
    cpu_zip = shutil.make_archive(str(cpu_output_dir), "zip", root_dir=str(cpu_output_dir))
    print("CPU results archive:", cpu_zip)
